# 03 — Leakage-safe time splits and preprocessing

**Objectives**

- Verify temporal and identity separation.
- Watch the feature contract reject a post-outcome field.
- Fit imputation, scaling, and encoding only through a training Pipeline.

**Prerequisite:** lesson 02 has prepared the dataset.


In [ ]:
import pandas as pd

from aai_local_classification.contracts import SplitName
from aai_local_classification.data import (
    add_intentional_leakage,
    load_split,
    prepare_dataset,
    validate_feature_contract,
)
from aai_local_classification.modeling import (
    build_candidate,
    candidate_specs,
    feature_frame,
)
from aai_local_classification.tracking import local_paths
from aai_local_classification.learning import study_root
from aai_local_classification.settings import load_settings

settings = load_settings()
root = study_root()
print(f"Course state: {root}")
print(f"Experiment: {settings.experiment_name}")


In [ ]:
paths = local_paths(root)
prepare_dataset(settings, paths.data_root)
train = load_split(settings, SplitName.TRAIN, paths.data_root)
validation = load_split(settings, SplitName.VALIDATION, paths.data_root)
pd.DataFrame(
    [
        ("train", train.snapshot_date.min(), train.snapshot_date.max(), len(train)),
        (
            "validation",
            validation.snapshot_date.min(),
            validation.snapshot_date.max(),
            len(validation),
        ),
    ],
    columns=["split", "first_snapshot", "last_snapshot", "rows"],
)


In [ ]:
assert train.snapshot_date.max() < validation.snapshot_date.min()
assert set(train.account_id).isdisjoint(validation.account_id)
print("Time order and account identity separation verified.")


In [ ]:
leaked = add_intentional_leakage(train)
unsafe_features = settings.features.model_copy(
    update={"categorical": settings.features.categorical + ("cancellation_reason",)}
)
unsafe_settings = settings.model_copy(update={"features": unsafe_features})
try:
    validate_feature_contract(unsafe_settings)
except ValueError as error:
    print(f"Blocked as intended: {error}")
else:
    raise AssertionError("The leakage demonstration should have been blocked")
assert "cancellation_reason" in leaked


In [ ]:
spec = candidate_specs()[0]
pipeline = build_candidate(spec, settings)
x_train = feature_frame(train, settings)
y_train = train[settings.data.target_column]
pipeline.fit(x_train, y_train)
transformed_columns = pipeline.named_steps["preprocess"].get_feature_names_out()
print(f"Declared input columns: {x_train.shape[1]}")
print(f"Post-encoding columns learned from train: {len(transformed_columns)}")
print(transformed_columns[:12])


Because the transformers and estimator are one Pipeline, a fit on a training
fold also fits the imputer/scaler/encoder only on that fold. Running preprocessing
once on all rows before a split would leak distribution information.

### Exercise

Why is a random stratified split not automatically “safer” than this time split?

**Hint:** the correct split imitates how the model will encounter unseen data.
If production predicts later cohorts, random mixing can hide temporal change.

**Checkpoint:** no test file was loaded, the prediction-time audit blocks the
teaching leakage field, and all learned preprocessing lives inside the Pipeline.

Next: **04_baseline.ipynb**.
